In [ ]:
# Run this cell - it will open a file upload dialog
from google.colab import files


In [ ]:
print("Select train_transaction.csv")
uploaded_txn = files.upload()

Select train_transaction.csv


Saving train_transaction.csv to train_transaction.csv


In [ ]:
print("Select train_identity.csv")
uploaded_id = files.upload()

Select train_identity.csv


Saving train_identity.csv to train_identity.csv


In [ ]:
import pandas as pd
import os

# Check if files exist
print("Files in current directory:")
for file in os.listdir():
    if file.endswith('.csv'):
        print(f"  - {file} ({os.path.getsize(file) / 1024 / 1024:.2f} MB)")

Files in current directory:
  - train_identity.csv (25.30 MB)
  - train_transaction.csv (651.69 MB)


In [ ]:
# Load both CSV files
print("Loading train_transaction.csv...")
train_txn = pd.read_csv('train_transaction.csv')

print("Loading train_identity.csv...")
train_id = pd.read_csv('train_identity.csv')

print(f"Transactions shape: {train_txn.shape}")
print(f"Identity shape: {train_id.shape}")
print(f"Fraud rate: {train_txn['isFraud'].mean()*100:.2f}%")

Loading train_transaction.csv...
Loading train_identity.csv...
Transactions shape: (590540, 394)
Identity shape: (144233, 41)
Fraud rate: 3.50%


In [ ]:
# Merge on TransactionID
print("Merging tables...")
df = train_txn.merge(train_id, on='TransactionID', how='left')
print(f"Merged shape: {df.shape}")

Merging tables...
Merged shape: (590540, 434)


In [ ]:
# Run this cell
import numpy as np

def reduce_memory(df):
    """Reduce memory usage of dataframe by converting dtypes"""
    print("Optimizing memory usage...")
    start_mem = df.memory_usage().sum() / 1024 / 1024
    print(f"Starting memory: {start_mem:.2f} MB")

    for col in df.columns:
        col_type = df[col].dtype

        # Skip object columns (strings)
        if col_type != 'object':
            # Convert int to smaller int
            if col_type != 'datetime64[ns]':
                if str(col_type)[:3] == 'int':
                    c_min = df[col].min()
                    c_max = df[col].max()
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                else:
                    # Convert float to smaller float
                    c_min = df[col].min()
                    c_max = df[col].max()
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        df[col] = df[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df[col] = df[col].astype(np.float32)

    end_mem = df.memory_usage().sum() / 1024 / 1024
    print(f"Final memory: {end_mem:.2f} MB")
    print(f"Reduced by: {(start_mem - end_mem):.2f} MB ({(1 - end_mem/start_mem)*100:.1f}%)")

    return df

# Apply memory reduction
df = reduce_memory(df)

Optimizing memory usage...
Starting memory: 1955.37 MB


/tmp/ipykernel_1506/3671304177.py:30: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1506/3671304177.py:30: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1506/3671304177.py:30: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1506/3671304177.py:30: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1506/3671304177.py:30: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1506/3671304177.py:30: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1506/3671304177.py:30: RuntimeW

Final memory: 645.97 MB
Reduced by: 1309.40 MB (67.0%)


In [ ]:
# Run this cell
print("Checking missing values...")
missing_before = df.isnull().sum().sum()
print(f"Total missing values before: {missing_before:,}")

# Fill all missing values with -999
df = df.fillna(-999)

missing_after = df.isnull().sum().sum()
print(f"Total missing values after: {missing_after:,}")
print(f"✓ All missing values handled")

Checking missing values...
Total missing values before: 115,523,073
Total missing values after: 0
✓ All missing values handled


In [ ]:
# Step 7: Separate Features and Target
y = df['isFraud']
X = df.drop(['isFraud'], axis=1)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Fraud rate: {y.mean()*100:.2f}%")

Features shape: (590540, 433)
Target shape: (590540,)
Fraud rate: 3.50%


In [ ]:
# Step 8: Remove TransactionID
if 'TransactionID' in X.columns:
    X = X.drop(['TransactionID'], axis=1)
    print("✓ Removed TransactionID")

print(f"Features remaining: {X.shape[1]}")

✓ Removed TransactionID
Features remaining: 432


In [ ]:
# Step 9: Encode Categorical Features
from sklearn.preprocessing import LabelEncoder

print("Encoding categorical features...")
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"Categorical columns found: {len(categorical_cols)}")

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = X[col].astype(str)
    X[col] = le.fit_transform(X[col])

print(f"✓ All categorical columns encoded")
print(f"Final feature shape: {X.shape}")

Encoding categorical features...
Categorical columns found: 31
✓ All categorical columns encoded
Final feature shape: (590540, 432)


In [ ]:
# Step 10: Feature Engineering - Time Features
if 'TransactionDT' in X.columns:
    print("Extracting time features from TransactionDT...")

    X['hour_of_day'] = (X['TransactionDT'] // 3600) % 24
    X['day_of_week'] = (X['TransactionDT'] // 86400) % 7
    X['day_of_month'] = (X['TransactionDT'] // 86400) % 30

    print(f"✓ Added: hour_of_day, day_of_week, day_of_month")
    print(f"Features now: {X.shape[1]}")
else:
    print("TransactionDT not found")

Extracting time features from TransactionDT...
✓ Added: hour_of_day, day_of_week, day_of_month
Features now: 435


/tmp/ipykernel_1506/3999537882.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['hour_of_day'] = (X['TransactionDT'] // 3600) % 24
/tmp/ipykernel_1506/3999537882.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['day_of_week'] = (X['TransactionDT'] // 86400) % 7
/tmp/ipykernel_1506/3999537882.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a

In [ ]:
# Step 11: Train-Test Split
from sklearn.model_selection import train_test_split

print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training: {X_train.shape[0]:,} samples")
print(f"Test: {X_test.shape[0]:,} samples")
print(f"Train fraud rate: {y_train.mean()*100:.2f}%")
print(f"Test fraud rate: {y_test.mean()*100:.2f}%")

Splitting data...
Training: 472,432 samples
Test: 118,108 samples
Train fraud rate: 3.50%
Test fraud rate: 3.50%


In [ ]:
# Step 12: Verify Data is Ready
print("=" * 50)
print("DATA PREPARATION COMPLETE")
print("=" * 50)
print(f"✓ Features: {X.shape[1]}")
print(f"✓ Training samples: {X_train.shape[0]:,}")
print(f"✓ Test samples: {X_test.shape[0]:,}")
print(f"✓ Missing values in training: {X_train.isnull().sum().sum()}")
print(f"✓ Missing values in test: {X_test.isnull().sum().sum()}")
print("\n✅ Ready for model training!")

DATA PREPARATION COMPLETE
✓ Features: 435
✓ Training samples: 472,432
✓ Test samples: 118,108
✓ Missing values in training: 0
✓ Missing values in test: 0

✅ Ready for model training!


In [ ]:
# Run this cell first
!pip install xgboost lightgbm -q
print("✓ Libraries installed")

✓ Libraries installed


In [ ]:
# Run this cell
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported")

✓ All libraries imported


In [ ]:
# Run this cell
print("=" * 50)
print("MODEL 1: LOGISTIC REGRESSION (BASELINE)")
print("=" * 50)

# Train the model
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test)
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

# ROC-AUC
roc_auc_lr = roc_auc_score(y_test, y_proba_lr)
print(f"ROC-AUC Score: {roc_auc_lr:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)
print(f"\nConfusion Matrix:")
print(f"True Negatives: {cm[0,0]:,} | False Positives: {cm[0,1]:,}")
print(f"False Negatives: {cm[1,0]:,} | True Positives: {cm[1,1]:,}")

MODEL 1: LOGISTIC REGRESSION (BASELINE)

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.80      0.88    113975
           1       0.10      0.62      0.17      4133

    accuracy                           0.79    118108
   macro avg       0.54      0.71      0.53    118108
weighted avg       0.95      0.79      0.86    118108

ROC-AUC Score: 0.7780

Confusion Matrix:
True Negatives: 91,251 | False Positives: 22,724
False Negatives: 1,583 | True Positives: 2,550


In [ ]:
# Run this cell
print("=" * 50)
print("MODEL 2: XGBOOST (MAIN MODEL)")
print("=" * 50)

# Calculate scale_pos_weight for imbalance
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

# Train XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

# ROC-AUC
roc_auc_xgb = roc_auc_score(y_test, y_proba_xgb)
print(f"ROC-AUC Score: {roc_auc_xgb:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_xgb)
print(f"\nConfusion Matrix:")
print(f"True Negatives: {cm[0,0]:,} | False Positives: {cm[0,1]:,}")
print(f"False Negatives: {cm[1,0]:,} | True Positives: {cm[1,1]:,}")

MODEL 2: XGBOOST (MAIN MODEL)

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.89      0.94    113975
           1       0.22      0.80      0.34      4133

    accuracy                           0.89    118108
   macro avg       0.60      0.85      0.64    118108
weighted avg       0.96      0.89      0.92    118108

ROC-AUC Score: 0.9238

Confusion Matrix:
True Negatives: 101,966 | False Positives: 12,009
False Negatives: 823 | True Positives: 3,310


In [ ]:
# Run this cell
print("=" * 50)
print("MODEL 3: LIGHTGBM (COMPARISON)")
print("=" * 50)

# Train LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train, y_train)

# Predictions
y_pred_lgb = lgb_model.predict(X_test)
y_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgb))

# ROC-AUC
roc_auc_lgb = roc_auc_score(y_test, y_proba_lgb)
print(f"ROC-AUC Score: {roc_auc_lgb:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lgb)
print(f"\nConfusion Matrix:")
print(f"True Negatives: {cm[0,0]:,} | False Positives: {cm[0,1]:,}")
print(f"False Negatives: {cm[1,0]:,} | True Positives: {cm[1,1]:,}")

MODEL 3: LIGHTGBM (COMPARISON)

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.89      0.94    113975
           1       0.21      0.81      0.33      4133

    accuracy                           0.88    118108
   macro avg       0.60      0.85      0.63    118108
weighted avg       0.96      0.88      0.92    118108

ROC-AUC Score: 0.9230

Confusion Matrix:
True Negatives: 101,163 | False Positives: 12,812
False Negatives: 778 | True Positives: 3,355


In [ ]:
# Run this cell
print("=" * 50)
print("MODEL COMPARISON SUMMARY")
print("=" * 50)

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost', 'LightGBM'],
    'ROC-AUC': [roc_auc_lr, roc_auc_xgb, roc_auc_lgb]
})

print(results.to_string(index=False))
print("\n" + "=" * 50)

# Find best model
best_model = results.loc[results['ROC-AUC'].idxmax(), 'Model']
best_score = results['ROC-AUC'].max()
print(f"🏆 BEST MODEL: {best_model} with ROC-AUC = {best_score:.4f}")

MODEL COMPARISON SUMMARY
              Model  ROC-AUC
Logistic Regression 0.777970
            XGBoost 0.923796
           LightGBM 0.923018

🏆 BEST MODEL: XGBoost with ROC-AUC = 0.9238


In [ ]:
# Run this cell
import joblib

# Save XGBoost as the main model
joblib.dump(xgb_model, 'fraud_model.pkl')
print("✓ XGBoost model saved as 'fraud_model.pkl'")

# Save feature names for API
feature_names = X_train.columns.tolist()
joblib.dump(feature_names, 'feature_names.pkl')
print("✓ Feature names saved")

# Save model metadata
model_metadata = {
    'model_type': 'XGBoost',
    'roc_auc': 0.9238,
    'fraud_recall': 0.80,
    'features_count': len(feature_names),
    'training_samples': len(X_train)
}
joblib.dump(model_metadata, 'model_metadata.pkl')
print("✓ Model metadata saved")

print("\n" + "=" * 50)
print("✅ MODEL TRAINING COMPLETE!")
print("=" * 50)
print(f"Model: XGBoost")
print(f"ROC-AUC: 0.9238")
print(f"Fraud Recall: 80%")
print(f"Features: {len(feature_names)}")
print("\nReady for API deployment! 🚀")

✓ XGBoost model saved as 'fraud_model.pkl'
✓ Feature names saved
✓ Model metadata saved

✅ MODEL TRAINING COMPLETE!
Model: XGBoost
ROC-AUC: 0.9238
Fraud Recall: 80%
Features: 435

Ready for API deployment! 🚀


In [ ]:
os.makedirs('backend', exist_ok=True)

In [ ]:
# Run this FULL cell - it creates the backend/main.py file
api_code = '''from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import numpy as np
import joblib
from typing import List, Dict, Any
import os

# Load model and artifacts
model_path = os.path.join('models', 'fraud_model.pkl')
features_path = os.path.join('models', 'feature_names.pkl')
metadata_path = os.path.join('models', 'model_metadata.pkl')

model = joblib.load(model_path)
feature_names = joblib.load(features_path)
metadata = joblib.load(metadata_path)

# Create FastAPI app
app = FastAPI(
    title="FraudShield AI - Fraud Detection API",
    description="Real-time fraud detection using XGBoost (92.38% ROC-AUC)",
    version="1.0.0"
)

# Define request schema
class TransactionRequest(BaseModel):
    features: Dict[str, float]

class BatchTransactionRequest(BaseModel):
    transactions: List[Dict[str, float]]

# Define response schema
class FraudResponse(BaseModel):
    fraud_prediction: bool
    fraud_probability: float
    risk_level: str
    confidence: float

# Health check endpoint
@app.get("/")
def root():
    return {
        "message": "FraudShield AI API",
        "status": "online",
        "model": metadata.get('model_type', 'XGBoost'),
        "roc_auc": metadata.get('roc_auc', 0.9238)
    }

# Model info endpoint
@app.get("/model-info")
def model_info():
    return {
        "model_type": metadata.get('model_type', 'XGBoost'),
        "roc_auc": metadata.get('roc_auc', 0.9238),
        "fraud_recall": metadata.get('fraud_recall', 0.80),
        "features_count": len(feature_names),
        "features": feature_names[:10]
    }

# Single prediction endpoint
@app.post("/predict", response_model=FraudResponse)
def predict(transaction: TransactionRequest):
    try:
        features_list = []
        for feature in feature_names:
            value = transaction.features.get(feature, -999)
            features_list.append(value)

        input_array = np.array(features_list).reshape(1, -1)

        prediction = model.predict(input_array)[0]
        probability = model.predict_proba(input_array)[0][1]

        if probability >= 0.7:
            risk_level = "HIGH"
        elif probability >= 0.4:
            risk_level = "MEDIUM"
        else:
            risk_level = "LOW"

        return FraudResponse(
            fraud_prediction=bool(prediction),
            fraud_probability=float(probability),
            risk_level=risk_level,
            confidence=float(probability)
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Batch prediction endpoint
@app.post("/predict-batch")
def predict_batch(transactions: BatchTransactionRequest):
    try:
        results = []
        for transaction in transactions.transactions:
            features_list = []
            for feature in feature_names:
                value = transaction.get(feature, -999)
                features_list.append(value)

            input_array = np.array(features_list).reshape(1, -1)
            prediction = model.predict(input_array)[0]
            probability = model.predict_proba(input_array)[0][1]

            if probability >= 0.7:
                risk_level = "HIGH"
            elif probability >= 0.4:
                risk_level = "MEDIUM"
            else:
                risk_level = "LOW"

            results.append({
                "fraud_prediction": bool(prediction),
                "fraud_probability": float(probability),
                "risk_level": risk_level
            })

        return {"predictions": results}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Write the API file
with open('backend/main.py', 'w') as f:
    f.write(api_code)

print("✓ FastAPI app created at backend/main.py")

✓ FastAPI app created at backend/main.py


In [ ]:
# Run this cell
requirements = '''fastapi==0.104.1
uvicorn==0.24.0
pandas==2.1.3
numpy==1.24.3
scikit-learn==1.3.0
xgboost==2.0.3
joblib==1.3.2
python-multipart==0.0.6
pydantic==2.5.0
'''

with open('backend/requirements.txt', 'w') as f:
    f.write(requirements)

print("✓ requirements.txt created")

✓ requirements.txt created


In [ ]:
# Run this cell
test_script = '''import requests
import json

API_URL = "http://localhost:8000"

def test_health():
    response = requests.get(f"{API_URL}/")
    print(f"Health Check: {response.json()}")
    return response.status_code == 200

def test_prediction():
    test_features = {
        "TransactionDT": 86400,
        "TransactionAmt": 150.50,
        "card1": 1,
        "card2": 2,
        "hour_of_day": 14,
        "day_of_week": 3,
        "day_of_month": 15
    }

    response = requests.post(f"{API_URL}/predict", json={"features": test_features})
    print(f"Prediction Response: {response.json()}")
    return response.status_code == 200

if __name__ == "__main__":
    print("Testing FraudShield AI API...")

    if test_health():
        print("✅ Health check passed")
    else:
        print("❌ Health check failed")

    if test_prediction():
        print("✅ Prediction test passed")
    else:
        print("❌ Prediction test failed")
'''

with open('backend/test_api.py', 'w') as f:
    f.write(test_script)

print("✓ test_api.py created")

✓ test_api.py created


In [ ]:
# Run this cell
!pip install fastapi uvicorn pandas numpy scikit-learn xgboost joblib python-multipart pydantic -q

print("✓ Dependencies installed")

✓ Dependencies installed


In [ ]:
# Run this cell to find your model files
import os

print("Looking for model files in current directory:")
for file in os.listdir('.'):
    if 'pkl' in file:
        print(f"  📄 {file}")

print("\nLooking for models folder:")
if os.path.exists('models'):
    print("  models folder exists")
    for file in os.listdir('models'):
        print(f"    📄 {file}")
else:
    print("  ❌ models folder does NOT exist")

Looking for model files in current directory:
  📄 feature_names.pkl
  📄 fraud_model.pkl
  📄 model_metadata.pkl

Looking for models folder:
  ❌ models folder does NOT exist


In [ ]:
# Run this cell
import os
import shutil

# Create models folder
os.makedirs('models', exist_ok=True)
print("✓ models folder created")

# Move model files to models folder
shutil.move('fraud_model.pkl', 'models/fraud_model.pkl')
print("✓ fraud_model.pkl moved to models/")

shutil.move('feature_names.pkl', 'models/feature_names.pkl')
print("✓ feature_names.pkl moved to models/")

shutil.move('model_metadata.pkl', 'models/model_metadata.pkl')
print("✓ model_metadata.pkl moved to models/")

print("\n✅ All model files moved to /content/models/")

✓ models folder created
✓ fraud_model.pkl moved to models/
✓ feature_names.pkl moved to models/
✓ model_metadata.pkl moved to models/

✅ All model files moved to /content/models/


In [ ]:
# Run this cell to verify
import os

print("📁 Files in /content/backend/:")
if os.path.exists('backend'):
    for file in os.listdir('backend'):
        print(f"   📄 {file}")
else:
    print("   ❌ backend folder not found")

print("\n📁 Files in /content/models/:")
if os.path.exists('models'):
    for file in os.listdir('models'):
        print(f"   📄 {file}")
else:
    print("   ❌ models folder not found")

📁 Files in /content/backend/:
   📄 requirements.txt
   📄 test_api.py
   📄 main.py

📁 Files in /content/models/:
   📄 feature_names.pkl
   📄 fraud_model.pkl
   📄 model_metadata.pkl


In [ ]:
# Run this cell
!pip install fastapi uvicorn pandas numpy scikit-learn xgboost joblib python-multipart pydantic -q

print("✓ Dependencies installed")

✓ Dependencies installed


In [ ]:
# Run this cell to start the API server in the background
import subprocess
import threading
import time

# Function to run the API server
def run_api():
    subprocess.run(["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"])

# Start server in background thread
api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

# Wait for server to start
time.sleep(5)

print("✓ API server started at http://localhost:8000")

✓ API server started at http://localhost:8000


In [ ]:
# Run this cell to test the API
import requests

print("=" * 50)
print("TESTING FRAUDSHIELD AI API")
print("=" * 50)

# Test 1: Health check
print("\n1. Health Check:")
response = requests.get("http://localhost:8000/")
print(f"   Status: {response.status_code}")
print(f"   Response: {response.json()}")

# Test 2: Model info
print("\n2. Model Info:")
response = requests.get("http://localhost:8000/model-info")
print(f"   Status: {response.status_code}")
data = response.json()
print(f"   Model: {data.get('model_type')}")
print(f"   ROC-AUC: {data.get('roc_auc')}")
print(f"   Features: {data.get('features_count')}")

# Test 3: Prediction
print("\n3. Fraud Prediction Test:")
test_transaction = {
    "features": {
        "TransactionDT": 86400,
        "TransactionAmt": 1500.00,  # High amount
        "card1": 1,
        "card2": 2,
        "hour_of_day": 3,  # Late night
        "day_of_week": 6   # Weekend
    }
}

response = requests.post("http://localhost:8000/predict", json=test_transaction)
print(f"   Status: {response.status_code}")
result = response.json()
print(f"   Fraud Prediction: {result.get('fraud_prediction')}")
print(f"   Fraud Probability: {result.get('fraud_probability')}")
print(f"   Risk Level: {result.get('risk_level')}")

print("\n" + "=" * 50)
print("✅ API TESTING COMPLETE!")
print("=" * 50)

TESTING FRAUDSHIELD AI API

1. Health Check:
   Status: 200
   Response: {'message': 'FraudShield AI API', 'status': 'online', 'model': 'XGBoost', 'roc_auc': 0.9238}

2. Model Info:
   Status: 200
   Model: XGBoost
   ROC-AUC: 0.9238
   Features: 435

3. Fraud Prediction Test:
   Status: 200
   Fraud Prediction: True
   Fraud Probability: 0.8336607813835144
   Risk Level: HIGH

✅ API TESTING COMPLETE!


In [ ]:
# Run this cell for another test
print("Testing with a legitimate-looking transaction:")
test_transaction_2 = {
    "features": {
        "TransactionDT": 86400,
        "TransactionAmt": 25.50,  # Small amount
        "card1": 1,
        "card2": 2,
        "hour_of_day": 14,  # Afternoon
        "day_of_week": 2    # Mid-week
    }
}

response = requests.post("http://localhost:8000/predict", json=test_transaction_2)
result = response.json()
print(f"   Fraud Prediction: {result.get('fraud_prediction')}")
print(f"   Fraud Probability: {result.get('fraud_probability')}")
print(f"   Risk Level: {result.get('risk_level')}")

Testing with a legitimate-looking transaction:


NameError: name 'requests' is not defined

In [ ]:
# Run this in Colab to download all files to your Mac
from google.colab import files

# Download model
files.download('models/fraud_model.pkl')

# Download feature names
files.download('models/feature_names.pkl')

# Download metadata
files.download('models/model_metadata.pkl')

# Download backend files
files.download('backend/main.py')
files.download('backend/requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>